# F5 — Fine-tuning & Model Adaptation · Hands-On Lab Notebook

**AI Engineering Specialist Track · Foundation Module F5**

Three labs, ~65 minutes, fully runnable offline:

1. **Prepare a fine-tuning dataset** — build and validate an instruction-tuning dataset (chat-format JSONL), check label balance, and split train/validation.
2. **Fine-tune: full vs LoRA** — train an adapter with gradient descent, then do the same with a low-rank **LoRA** adapter and compare trainable-parameter counts.
3. **Evaluate & decide** — base vs fine-tuned accuracy, an overfitting / early-stopping demo, and the prompt-vs-RAG-vs-fine-tune decision.

---

### A note on how this notebook runs

The labs teach the **real** fine-tuning workflow — dataset formatting, the LoRA low-rank
decomposition, the gradient-descent training loop, and evaluation including overfitting.

To guarantee it runs **with zero errors and zero warnings on any machine**, it uses a small,
self-contained model in pure NumPy instead of a GPU + transformers + PEFT. A frozen random
projection stands in for a pretrained representation; the trainable adapter and its LoRA
factorisation are implemented and trained for real. The *Case Analysis Solutions* doc maps
every stand-in to its production equivalent (Hugging Face PEFT, `SFTTrainer`, etc.). In a
real project you swap the frozen base and the trainer; the concepts are identical.

## Setup

Only NumPy, Pandas, and Matplotlib — no GPU, no model downloads, no API keys.

In [ ]:
# Clean, quiet environment ----------------------------------------------------
import os, warnings, logging, re, json, math, random
from dataclasses import dataclass, field

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
logging.getLogger("matplotlib").setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RNG = np.random.default_rng(42)
random.seed(42)
print("Environment ready.")
print("NumPy:", np.__version__, "| Pandas:", pd.__version__)

---
# Lab 1 — Prepare a fine-tuning dataset

**Goal:** build a clean **instruction-tuning dataset** that adapts a base model to route
telecom support tickets into five intents. In real fine-tuning, *the dataset is the product*
— quality, balance, and format matter more than any hyperparameter.

### Step 1.1 — Synthesise a labelled dataset

We generate support tickets for five intents from templated stems with variation. Each
example is a `(text, intent)` pair. (In practice this is where you'd curate real tickets;
the discipline below — balance, dedup, splitting — is identical.)

In [ ]:
INTENTS = ["billing_dispute", "technical_support", "cancellation",
           "plan_change", "account_access"]

# Every template carries at least one {slot}, so filling yields many unique variants
# per class -> we can draw a BALANCED number of distinct examples for each intent.
_TEMPLATES = {
    "billing_dispute": [
        "I was charged {amt} that I never authorised on my {month} bill",
        "There is a duplicate charge for {svc} on {month}, I want it refunded",
        "Why is my bill {amt} higher this {month}, I see an unexpected fee",
        "Dispute the {amt} roaming charge added to {svc} {when}",
        "I was billed twice for {svc}, please refund the {amt}",
        "An unexpected {amt} fee for {svc} appeared {when}",
    ],
    "technical_support": [
        "My internet keeps dropping on {dev} every few minutes {when}",
        "No signal on {dev} {when}, nothing works",
        "The router restarts on its own and {dev} is slow {when}",
        "My broadband is unusably slow on {dev} during the evening {when}",
        "Calls keep failing and data will not connect on {dev}",
        "{dev} loses connection {when} and will not reconnect",
    ],
    "cancellation": [
        "I want to cancel my service effective {month}",
        "How do I close my account permanently from {dev}",
        "Please terminate my subscription {when}, I am switching providers",
        "Cancel my {svc}, I no longer need it as of {month}",
        "I would like to end my contract {when} and return the equipment",
        "Stop my {svc} from {month}, I am leaving",
    ],
    "plan_change": [
        "I want to upgrade to a bigger data plan from {month}",
        "Can I switch to the unlimited plan starting {month}",
        "Downgrade me to the basic plan {when} to save money",
        "Change my {svc} to add more data from {month}",
        "Move me to a cheaper plan with fewer minutes {when}",
        "Upgrade {svc} to a larger allowance starting {month}",
    ],
    "account_access": [
        "I cannot log in on {dev}, my password is not working {when}",
        "I forgot my password and need to reset it on {dev}",
        "My account is locked on {dev} after too many attempts {when}",
        "Reset my login credentials, I am locked out of {dev}",
        "I am not receiving the password reset email {when}",
        "Cannot access my account from {dev} {when}, login keeps failing",
    ],
}
_FILL = {
    "{amt}":   ["$25", "$40", "$19.99", "$120", "$60", "$15", "$220"],
    "{month}": ["January", "March", "last month", "this month", "April", "February", "May"],
    "{svc}":   ["broadband", "the mobile plan", "the TV package", "international calls",
                "the data add-on", "the home line"],
    "{when}":  ["this morning", "since yesterday", "for two days", "since the update",
                "all week", "since Monday"],
    "{dev}":   ["my phone", "the app", "my laptop", "the web portal", "my tablet", "the dashboard"],
}

def _fill_one(t, rng):
    for slot, opts in _FILL.items():
        while slot in t:
            t = t.replace(slot, rng.choice(opts), 1)
    return t

PER_CLASS = 22
gen_rng = np.random.default_rng(42)
records = []
for intent, templates in _TEMPLATES.items():
    seen = set()
    attempts = 0
    while len(seen) < PER_CLASS and attempts < 2000:
        text = _fill_one(templates[gen_rng.integers(len(templates))], gen_rng)
        attempts += 1
        if text not in seen:
            seen.add(text)
            records.append({"text": text, "intent": intent})

df = pd.DataFrame(records).reset_index(drop=True)
print(f"Generated {len(df)} unique examples, {PER_CLASS} per intent "
      f"across {df['intent'].nunique()} intents (balanced by construction).")
df.head(5)

### Step 1.2 — Validate: label balance & basic quality checks

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.3))
ax.bar(counts.index, counts.values, color="#0E7490")
ax.set_title("Lab 1 — class balance of the fine-tuning set", fontsize=12, fontweight="bold")
ax.set_ylabel("examples")
ax.tick_params(axis="x", rotation=20)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
fig.tight_layout()
plt.show()

### Step 1.3 — Format as chat-style JSONL (the real fine-tuning format)

Instruction/chat fine-tuning expects a list of messages per example. This is exactly the
shape Hugging Face `SFTTrainer` and the OpenAI fine-tuning API consume.

### Step 1.4 — Stratified train/validation split

In [ ]:
def stratified_split(frame, label_col="intent", val_frac=0.25, seed=42):
    rng = np.random.default_rng(seed)
    train_idx, val_idx = [], []
    for _, grp in frame.groupby(label_col):
        idx = grp.index.to_numpy().copy()
        rng.shuffle(idx)
        n_val = max(1, int(round(len(idx) * val_frac)))
        val_idx.extend(idx[:n_val]); train_idx.extend(idx[n_val:])
    return frame.loc[train_idx].reset_index(drop=True), frame.loc[val_idx].reset_index(drop=True)

train_df, val_df = stratified_split(df)
print(f"Train: {len(train_df)} examples | Validation: {len(val_df)} examples")
print("\nPer-class counts (train / val):")
for intent in INTENTS:
    print(f"  {intent:<18} {int((train_df.intent==intent).sum()):>3} / "
          f"{int((val_df.intent==intent).sum()):>2}")

### Lab 1 — What to look for (discussion)

1. **The dataset is the product.** Most fine-tuning failures are data failures — imbalance, leakage, inconsistent labels — not bad hyperparameters.
2. **Balance matters.** A skewed label distribution teaches the model to over-predict the majority class. We checked the imbalance ratio before training.
3. **Format is a contract.** The chat-JSONL shape is what real trainers expect. Getting it wrong is the most common first-run error in production fine-tuning.
4. **Stratified splitting** keeps every class represented in validation — otherwise your eval metric is measuring the wrong thing.

---
# Lab 2 — Fine-tune: full adapter vs LoRA

**Goal:** adapt a *frozen* base representation to the routing task by training an adapter
with gradient descent — first a **full** weight matrix, then a **LoRA** low-rank
factorisation — and compare trainable-parameter counts.

### Step 2.1 — A frozen base representation

A real fine-tune starts from a frozen pretrained model. We simulate that: TF-IDF features →
a **frozen random projection** → `tanh`. This fixed map turns text into a dense vector `z`.
We never train it — exactly like freezing the base model's weights.

In [ ]:
_word_re = re.compile(r"\b\w+\b")

class FrozenBase:
    """Frozen featuriser: TF-IDF -> fixed random projection -> tanh. Stands in for a
    frozen pretrained model that emits a representation we adapt on top of."""
    def __init__(self, hidden=128, seed=0):
        self.hidden = hidden
        self.vocab, self.idf, self.R = {}, None, None
        self.mean_, self.std_ = None, None
        self._seed = seed

    def _tok(self, t): return _word_re.findall(t.lower())

    def fit(self, texts):
        dfc = {}
        for t in texts:
            for w in set(self._tok(t)):
                dfc[w] = dfc.get(w, 0) + 1
        self.vocab = {w: i for i, w in enumerate(sorted(dfc))}
        n = len(texts)
        self.idf = np.ones(len(self.vocab))
        for w, i in self.vocab.items():
            self.idf[i] = math.log((1 + n) / (1 + dfc[w])) + 1.0
        rng = np.random.default_rng(self._seed)
        self.R = rng.standard_normal((len(self.vocab), self.hidden)) / math.sqrt(len(self.vocab))
        Z = self._project(texts)
        self.mean_, self.std_ = Z.mean(0), Z.std(0) + 1e-8   # guard zero std
        return self

    def _tfidf(self, text):
        v = np.zeros(len(self.vocab))
        toks = self._tok(text)
        if not toks: return v
        for w in toks:
            j = self.vocab.get(w)
            if j is not None: v[j] += 1.0
        v /= len(toks); v *= self.idf
        return v

    def _project(self, texts):
        X = np.vstack([self._tfidf(t) for t in texts])
        return np.tanh(X @ self.R)

    def transform(self, texts):
        Z = self._project(texts)
        return (Z - self.mean_) / self.std_      # standardise with frozen stats

base = FrozenBase(hidden=128, seed=0).fit(train_df["text"].tolist())
Z_train = base.transform(train_df["text"].tolist())
Z_val   = base.transform(val_df["text"].tolist())
y_train = np.array([INTENTS.index(i) for i in train_df["intent"]])
y_val   = np.array([INTENTS.index(i) for i in val_df["intent"]])
H, K = base.hidden, len(INTENTS)
print(f"Frozen representation: train {Z_train.shape}, val {Z_val.shape}")
print(f"Hidden dim H = {H}, classes K = {K}")

### Step 2.2 — The training pieces (softmax head + residual adapter)

The model is `logits = (Z + Z @ dW) @ W_head + b`. The adapter `dW` (H×H) is the weight
matrix we "fine-tune"; the head maps to the K intents. We train with plain gradient descent
on cross-entropy. This is the real training loop, just small.

In [ ]:
def softmax(x):
    x = x - x.max(axis=1, keepdims=True)         # stability: no overflow
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

def cross_entropy(p, y):
    return float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1.0)).mean())

def accuracy(logits, y):
    return float((logits.argmax(1) == y).mean())

def forward(Z, dW, W_head, b):
    Hrep = Z + Z @ dW               # residual adaptation (LoRA-style: base + delta)
    logits = Hrep @ W_head + b
    return Hrep, logits

print("Training pieces ready (softmax, cross-entropy, residual-adapter forward pass).")

### Step 2.3 — Full fine-tuning (train the entire `dW`)

In [ ]:
def train_full(Z, y, Zv, yv, epochs=200, lr=0.5, seed=0):
    rng = np.random.default_rng(seed)
    dW = np.zeros((H, H))                          # adapter starts as a no-op
    W_head = rng.standard_normal((H, K)) * 0.01
    b = np.zeros(K)
    Y = np.eye(K)[y]
    hist = []
    for ep in range(epochs):
        Hrep, logits = forward(Z, dW, W_head, b)
        p = softmax(logits)
        dlogits = (p - Y) / len(y)
        dW_head = Hrep.T @ dlogits
        db = dlogits.sum(0)
        dHrep = dlogits @ W_head.T
        dDW = Z.T @ dHrep                          # gradient wrt the full adapter
        dW -= lr * dDW; W_head -= lr * dW_head; b -= lr * db
        if ep % 20 == 0 or ep == epochs - 1:
            _, lv = forward(Zv, dW, W_head, b)
            hist.append({"epoch": ep, "train_loss": cross_entropy(p, y),
                         "val_acc": accuracy(lv, yv)})
    return (dW, W_head, b), pd.DataFrame(hist)

full_params, full_hist = train_full(Z_train, y_train, Z_val, y_val)
full_trainable = H * H + H * K + K
_, full_logits_val = forward(Z_val, *full_params)
print(f"Full fine-tuning — trainable params: {full_trainable:,}")
print(f"Final train loss: {full_hist.iloc[-1]['train_loss']:.3f} | "
      f"val accuracy: {accuracy(full_logits_val, y_val):.3f}")

### Step 2.4 — LoRA fine-tuning (train a low-rank `dW = A @ B`)

Instead of the full H×H matrix, LoRA learns two thin matrices `A` (H×r) and `B` (r×H) with
rank `r ≪ H`. The adapter is their product. `B` starts at zero so the adapter begins as a
no-op — exactly how real LoRA initialises. Only `A` and `B` (and the head) are trained.

In [ ]:
def train_lora(Z, y, Zv, yv, r=4, epochs=200, lr=0.5, seed=0):
    rng = np.random.default_rng(seed)
    A = rng.standard_normal((H, r)) * 0.01
    B = np.zeros((r, H))                            # B=0 -> dW=0 at start (real LoRA init)
    W_head = rng.standard_normal((H, K)) * 0.01
    b = np.zeros(K)
    Y = np.eye(K)[y]
    hist = []
    for ep in range(epochs):
        dW = A @ B
        Hrep, logits = forward(Z, dW, W_head, b)
        p = softmax(logits)
        dlogits = (p - Y) / len(y)
        dW_head = Hrep.T @ dlogits
        db = dlogits.sum(0)
        dHrep = dlogits @ W_head.T
        dDW = Z.T @ dHrep                           # gradient wrt dW = A@B
        dA = dDW @ B.T                              # chain rule into the factors
        dB = A.T @ dDW
        A -= lr * dA; B -= lr * dB
        W_head -= lr * dW_head; b -= lr * db
        if ep % 20 == 0 or ep == epochs - 1:
            _, lv = forward(Zv, A @ B, W_head, b)
            hist.append({"epoch": ep, "train_loss": cross_entropy(p, y),
                         "val_acc": accuracy(lv, yv)})
    return (A, B, W_head, b), pd.DataFrame(hist), r

lora_params, lora_hist, r = train_lora(Z_train, y_train, Z_val, y_val, r=4)
A, B, W_head_l, b_l = lora_params
lora_trainable = H * r + r * H + H * K + K
_, lora_logits_val = forward(Z_val, A @ B, W_head_l, b_l)
print(f"LoRA fine-tuning (rank r={r}) — trainable params: {lora_trainable:,}")
print(f"Final train loss: {lora_hist.iloc[-1]['train_loss']:.3f} | "
      f"val accuracy: {accuracy(lora_logits_val, y_val):.3f}")
print(f"\nParameter efficiency: LoRA trains {full_trainable / lora_trainable:.1f}x "
      f"fewer adapter params than full fine-tuning, for comparable accuracy.")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.6))
ax.plot(full_hist["epoch"], full_hist["train_loss"], marker="o", color="#E0533D", label="Full fine-tune")
ax.plot(lora_hist["epoch"], lora_hist["train_loss"], marker="s", color="#0E7490", label="LoRA")
ax.set_title("Lab 2 — training loss: full vs LoRA", fontsize=12, fontweight="bold")
ax.set_xlabel("epoch"); ax.set_ylabel("train cross-entropy")
ax.legend(frameon=False)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
fig.tight_layout()
plt.show()

### Lab 2 — What to look for (discussion)

1. **The adapter starts as a no-op.** Both `dW` (full) and `A@B` (LoRA, with `B=0`) begin at zero, so training *adds* to the frozen base — this is the residual idea behind real LoRA.
2. **LoRA trains far fewer parameters.** Same task, comparable accuracy, a fraction of the trainable weights. At real scale (billions of frozen params) that fraction is what makes fine-tuning affordable on one GPU.
3. **Rank `r` is the key knob.** Small `r` = fewer params and more regularisation; too small and the adapter can't capture the task. We used `r=4`.
4. **The loss curve is the pulse.** A smoothly decreasing train loss means the optimiser is healthy; a flat or exploding curve means the learning rate or data is wrong.

---
# Lab 3 — Evaluate & decide

**Goal:** measure the fine-tune against an unadapted **base**, expose **overfitting** with a
train-vs-validation curve and early stopping, and place fine-tuning in the
**prompt → RAG → fine-tune** decision.

### Step 3.1 — Base vs fine-tuned accuracy

The "base" model is the frozen representation with an *untrained* head — what you get with no
adaptation (≈ chance for a balanced 5-class task). We compare it to the full and LoRA fine-tunes.

In [ ]:
rng = np.random.default_rng(7)
W_head_base = rng.standard_normal((H, K)) * 0.01
b_base = np.zeros(K)
_, base_logits_val = forward(Z_val, np.zeros((H, H)), W_head_base, b_base)

results = {
    "base (no fine-tune)": accuracy(base_logits_val, y_val),
    "full fine-tune":      accuracy(full_logits_val, y_val),
    f"LoRA (r={r})":       accuracy(lora_logits_val, y_val),
}
print("Validation accuracy:")
for k, v in results.items():
    print(f"  {k:<22} {v:.3f}")
print(f"\nChance baseline for {K} balanced classes: {1/K:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 3.6))
labels = list(results.keys()); vals = list(results.values())
bars = ax.bar(labels, vals, color=["#9AA5AD", "#E0533D", "#0E7490"])
ax.axhline(1/K, ls="--", lw=1, color="#64748B")
ax.text(2.4, 1/K + 0.02, "chance", fontsize=9, color="#64748B")
ax.set_ylim(0, 1.05); ax.set_ylabel("val accuracy")
ax.set_title("Lab 3 — adaptation lifts accuracy well above the base", fontsize=12, fontweight="bold")
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.02, f"{v:.2f}", ha="center", fontsize=10)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
fig.tight_layout()
plt.show()

### Step 3.2 — Overfitting & early stopping

To make overfitting visible on this small, clean dataset, we add a little **label noise** to
the training set (some labels are deliberately flipped) and train an over-powered adapter for
many epochs. The model memorises the noisy training labels — train loss keeps falling — while
**validation** loss bottoms out and then climbs. The best checkpoint is where validation is
lowest, which is where early stopping would cut.

In [ ]:
def train_track(Z, y, Zv, yv, epochs=300, lr=0.6, noise=0.20, seed=1):
    rng = np.random.default_rng(seed)
    # Inject label noise into a COPY of the training labels (val stays clean).
    y_noisy = y.copy()
    flip = rng.random(len(y)) < noise
    y_noisy[flip] = rng.integers(0, K, size=int(flip.sum()))
    dW = np.zeros((H, H)); W_head = rng.standard_normal((H, K)) * 0.01; b = np.zeros(K)
    Y = np.eye(K)[y_noisy]
    hist = []
    for ep in range(epochs):
        Hrep, logits = forward(Z, dW, W_head, b)
        p = softmax(logits)
        dlogits = (p - Y) / len(y_noisy)
        dW -= lr * (Z.T @ (dlogits @ W_head.T))
        W_head -= lr * (Hrep.T @ dlogits); b -= lr * dlogits.sum(0)
        _, lv = forward(Zv, dW, W_head, b)
        hist.append({"epoch": ep, "train_loss": cross_entropy(p, y_noisy),
                     "val_loss": cross_entropy(softmax(lv), yv)})
    return pd.DataFrame(hist)

track = train_track(Z_train, y_train, Z_val, y_val)
best = int(track.loc[track["val_loss"].idxmin(), "epoch"])
print(f"Lowest validation loss at epoch {best} of {len(track)-1} "
      f"-> that's where early stopping would cut.")
print(f"After that, train loss keeps falling to {track['train_loss'].iloc[-1]:.3f} "
      f"while val loss climbs to {track['val_loss'].iloc[-1]:.3f} (overfitting the noise).")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.6))
ax.plot(track["epoch"], track["train_loss"], color="#E0533D", label="train loss")
ax.plot(track["epoch"], track["val_loss"], color="#0E7490", label="validation loss")
ax.axvline(best, ls="--", lw=1, color="#64748B")
ax.text(best + 4, ax.get_ylim()[1]*0.8, f"early stop\n(epoch {best})", fontsize=9, color="#64748B")
ax.set_title("Lab 3 — overfitting: train falls, validation turns up", fontsize=12, fontweight="bold")
ax.set_xlabel("epoch"); ax.set_ylabel("cross-entropy")
ax.legend(frameon=False)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
fig.tight_layout()
plt.show()

### Step 3.3 — The decision: prompt vs RAG vs fine-tune

Fine-tuning is the heaviest lever. Reach for it only when the lighter ones fall short.

In [ ]:
decision = pd.DataFrame([
    {"approach": "Prompt engineering", "changes": "the instruction",
     "cost": "lowest", "best_for": "behaviour & format you can describe in words"},
    {"approach": "RAG (F4)", "changes": "the context",
     "cost": "low-medium", "best_for": "injecting fresh or private *knowledge*"},
    {"approach": "Fine-tuning (F5)", "changes": "the model weights",
     "cost": "highest", "best_for": "a consistent *skill/style/format* at scale"},
])
print(decision.to_string(index=False))
print("\nRule of thumb: prompt first, add RAG for knowledge, fine-tune for a durable skill")
print("the model should just *have* — and remember you can combine all three.")

### Lab 3 — What to look for (discussion)

1. **Adaptation works, and you can prove it.** Fine-tuned accuracy sits far above the chance-level base — the held-out validation set is what makes that claim credible.
2. **Overfitting is visible, not theoretical.** When train loss keeps dropping while validation loss climbs, the model is memorising. Early stopping takes the best validation checkpoint.
3. **LoRA ≈ full, cheaper.** Comparable accuracy at a fraction of the trainable parameters — the reason PEFT is the default in practice.
4. **Fine-tuning is the last lever, not the first.** Prompt for behaviour, RAG for knowledge, fine-tune for a durable skill. Most problems are solved before you reach weights.

**Next:** F6 — Evaluation, telemetry & observability at scale. How you measure all of F1–F5 in production.

---
# F5 Lab — Wrap-up checklist

- [ ] Why is the dataset, not the hyperparameters, where most fine-tunes succeed or fail?
- [ ] What does the chat-JSONL format represent, and who consumes it?
- [ ] In LoRA, what are `A` and `B`, and why does `B` start at zero?
- [ ] Why does LoRA train so many fewer parameters than full fine-tuning?
- [ ] What does the rank `r` control?
- [ ] How does a train-vs-validation curve reveal overfitting, and what is early stopping?
- [ ] When do you fine-tune instead of prompting or using RAG — and can you combine them?